In [ ]:
# Project path setup after moving notebooks into notebooks/
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
# Cell 5: 前 100 个 subaction 的 E0/E1/E2/E3 诊断实验
# ============================================================
# 目标：把 VRB 数据处理流程按 subaction 对齐，并用对照实验直观看出：
#   1. 当前策略卡在哪里；
#   2. 放宽 reference 搜索后是否改善；
#   3. 从 first contact 改成 all contact frames 后是否改善；
#   4. 120 帧窗口和 full previous search 的差距有多大。
#
# 输出目录：outputs/数据处理小批量测试前100个subaction/
# ============================================================

%matplotlib inline

from collections import Counter, defaultdict
from contextlib import redirect_stdout
import io
import json
import os
from pathlib import Path
import shutil

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from epic_kitchens.hoa import load_detections
from epic_kitchens.hoa.types import HandState

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

from vrbreproduction.contact_point_utils import (
    ContactExtractionConfig,
    get_active_hand_bbox,
    get_valid_object_bboxes,
    select_active_object_bbox,
)
from vrbreproduction.label_heatmap_utils import (
    build_label_heatmaps,
    draw_vrb_style_affordance_overlay,
    merge_label_heatmaps,
    save_label_heatmap_outputs,
    transform_covariances_by_homography,
)
from vrbreproduction.pipeline_retention import fit_cell2_contact_gmm, run_cell4_heatmap_gate, _json_safe
from vrbreproduction.problem3_runner import run_problem3_cell3
from vrbreproduction.problem3_utils import (
    build_dynamic_mask,
    compute_pairwise_homography,
    find_strict_humanless_frame,
    transform_points,
    transform_bbox_to_polygon,
    polygon_area,
    count_points_near_polygon,
    draw_problem3_full_overlay,
    draw_problem3_crop_overlay,
)


# ---- 实验配置：通常只改这里 ----
SUBACTION_VIDEO_ID = 'P01_109'
NUM_SUBACTIONS = 100
REFERENCE_WINDOW_BEFORE_START = 120
OUTPUT_ROOT = Path('outputs') / '数据处理小批量测试前100个subaction'
ANNOTATION_CSV = Path('data/annotations/epic-kitchens-100-annotations/EPIC_100_train.csv')
HOA_PKL = Path('data/P01_109.pkl')
IMAGE_DIR = Path('data/P01_109_frames')

# all-contact 的全量候选会先统计，但送进昂贵 Cell3/Cell4 的候选需要限量。
# 这里按时间均匀抽样，每个 subaction 最多深跑 5 个候选，避免大量相邻帧重复做 homography。
MAX_CANDIDATES_PER_SUBACTION = 5

EXPERIMENTS = [
    {
        'experiment_id': 'E0',
        'name': 'E0_current_first_contact_inside_subaction',
        'contact_strategy': 'first_contact',
        'reference_strategy': 'inside_subaction',
        'description': '当前失败基线：每个 subaction 只取 first contact，reference 只能在 subaction 内向前找。',
    },
    {
        'experiment_id': 'E1',
        'name': 'E1_first_contact_reference_window_120',
        'contact_strategy': 'first_contact',
        'reference_strategy': 'window_before_start_120',
        'description': '只放宽 reference 搜索范围，用来验证 no_strict_humanless_frame 是否由 subaction 起点限制导致。',
    },
    {
        'experiment_id': 'E2',
        'name': 'E2_all_contacts_reference_window_120',
        'contact_strategy': 'all_contacts',
        'reference_strategy': 'window_before_start_120',
        'description': '主方案：subaction 内扫描所有 contact frames，同时允许从 subaction 前 120 帧找 reference。',
    },
    {
        'experiment_id': 'E3',
        'name': 'E3_all_contacts_full_previous_reference',
        'contact_strategy': 'all_contacts',
        'reference_strategy': 'full_previous',
        'description': '上限对照：all contact frames，reference 可以从视频开头向前找。',
    },
]


def silent_call(func, *args, **kwargs):
    with redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)


class Problem3CachedRunner:
    """Notebook-local cache for expensive adjacent-frame homography work."""

    def __init__(self, detections, image_dir, output_root):
        self.detections = detections
        self.image_dir = Path(image_dir)
        self.output_root = Path(output_root)
        self.gray_cache = {}
        self.mask_cache = {}
        self.pair_cache = {}
        self.cache_stats = Counter()

    def load_gray(self, frame_idx):
        frame_idx = int(frame_idx)
        if frame_idx not in self.gray_cache:
            img_path = self.image_dir / f'frame_{frame_idx + 1:010d}.jpg'
            img = cv2.imread(str(img_path))
            if img is None:
                raise FileNotFoundError(f'missing frame image: {img_path}')
            self.gray_cache[frame_idx] = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return self.gray_cache[frame_idx]

    def load_bgr(self, frame_idx):
        img_path = self.image_dir / f'frame_{int(frame_idx) + 1:010d}.jpg'
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f'missing frame image: {img_path}')
        return img

    def dynamic_mask(self, frame_idx, hand_score_threshold=0.5, object_score_threshold=0.5):
        key = (int(frame_idx), float(hand_score_threshold), float(object_score_threshold))
        if key not in self.mask_cache:
            gray = self.load_gray(frame_idx)
            self.mask_cache[key] = build_dynamic_mask(
                self.detections[int(frame_idx)],
                gray.shape,
                hand_score_threshold,
                object_score_threshold,
            )
        return self.mask_cache[key]

    def pairwise_homography(self, cur_idx, prev_idx, hand_score_threshold=0.5, object_score_threshold=0.5):
        key = (int(cur_idx), int(prev_idx), float(hand_score_threshold), float(object_score_threshold))
        if key in self.pair_cache:
            self.cache_stats['pair_cache_hits'] += 1
            H, stats = self.pair_cache[key]
            return H.copy() if H is not None else None, dict(stats)

        cur_gray = self.load_gray(cur_idx)
        prev_gray = self.load_gray(prev_idx)
        cur_mask = self.dynamic_mask(cur_idx, hand_score_threshold, object_score_threshold)
        prev_mask = self.dynamic_mask(prev_idx, hand_score_threshold, object_score_threshold)
        H, stats = compute_pairwise_homography(
            prev_gray,
            cur_gray,
            prev_mask,
            cur_mask,
            nfeatures=2000,
            ratio_test=0.75,
            ransac_reproj_threshold=5.0,
        )
        self.cache_stats['pair_cache_misses'] += 1
        self.pair_cache[key] = (H.copy() if H is not None else None, dict(stats))
        return H, dict(stats)

    def accumulate_homography_to_ref(self, ref_idx, target_idx):
        ref_idx = int(ref_idx)
        target_idx = int(target_idx)
        if target_idx == ref_idx:
            return np.eye(3, dtype=np.float64), [], None
        if target_idx < ref_idx:
            return None, [], 'target_idx_less_than_ref_idx'

        H_total = np.eye(3, dtype=np.float64)
        pair_stats = []
        for cur_idx in range(target_idx, ref_idx, -1):
            prev_idx = cur_idx - 1
            H_cur_to_prev, stats = self.pairwise_homography(cur_idx, prev_idx)
            stats['cur'] = cur_idx
            stats['prev'] = prev_idx
            pair_stats.append(stats)

            if H_cur_to_prev is None:
                stats['passed_quality_gate'] = False
                return None, pair_stats, f'pairwise_homography_failed_f{cur_idx}_to_f{prev_idx}'
            if stats['good_matches'] < 60 or stats['inliers'] < 40 or stats['inlier_ratio'] < 0.45:
                stats['passed_quality_gate'] = False
                return None, pair_stats, f'pairwise_homography_low_quality_f{cur_idx}_to_f{prev_idx}'

            stats['passed_quality_gate'] = True
            H_total = H_cur_to_prev @ H_total
        return H_total, pair_stats, None

    def run_cell3(self, t_contact, active_hand, contact_means, output_dir, min_ref_frame_idx=0, discard=False, discard_reason=None):
        result = {
            'status': 'KEEP',
            'discard': bool(discard),
            'discard_reason': discard_reason,
            'ref_idx': None,
            'trajectory_pixels': [],
            'trajectory_missing_offsets': [],
            'all_pair_stats': [],
            'H_contact_to_ref': None,
            'mu_transformed': None,
            'tau_transformed': [],
            'object_polygon_ref': None,
            'hand_polygon_ref': None,
            'contact_centroid': None,
            'projected_area': None,
            'area_ratio': None,
            'inside_count': None,
            'centroid_dist': None,
            'full_overlay_path': None,
            'crop_path': None,
            'crop_bbox': None,
        }
        if result['discard']:
            result['status'] = 'DISCARD'
            return result
        if contact_means is None:
            result.update(discard=True, discard_reason='missing_contact_means', status='DISCARD')
            return result

        ref_idx, debug_info = find_strict_humanless_frame(
            self.detections,
            int(t_contact),
            score_threshold=0.5,
            min_no_hand_streak=3,
            min_frame_idx=int(min_ref_frame_idx),
        )
        result['ref_idx'] = ref_idx
        if ref_idx is None:
            result.update(discard=True, discard_reason='no_strict_humanless_frame', status='DISCARD')
            return result

        ref_img = self.load_bgr(ref_idx)
        h_img, w_img = ref_img.shape[:2]

        trajectory_pixels = []
        trajectory_missing_offsets = []
        for offset in range(6):
            idx = int(t_contact) + offset
            if idx >= len(self.detections):
                break
            frame_det = self.detections[idx]
            found_active_hand = False
            for hand in frame_det.hands:
                if hand.score > 0.5 and hand.side.name.lower() == active_hand:
                    bbox = [hand.bbox.left, hand.bbox.top, hand.bbox.right, hand.bbox.bottom]
                    cx = (bbox[0] + bbox[2]) / 2 * w_img
                    cy = (bbox[1] + bbox[3]) / 2 * h_img
                    trajectory_pixels.append(np.array([cx, cy], dtype=np.float32))
                    found_active_hand = True
                    break
            if not found_active_hand:
                trajectory_missing_offsets.append(offset)
        result['trajectory_pixels'] = trajectory_pixels
        result['trajectory_missing_offsets'] = trajectory_missing_offsets
        if len(trajectory_pixels) == 0:
            result.update(discard=True, discard_reason='missing_trajectory_points', status='DISCARD')
            return result

        cumulative_H_list = []
        all_pair_stats = []
        fail_reason = None
        homography_failed = False
        for offset in range(len(trajectory_pixels)):
            target_idx = int(t_contact) + offset
            H_target_to_ref, pair_stats, fail_reason = self.accumulate_homography_to_ref(ref_idx, target_idx)
            cumulative_H_list.append(H_target_to_ref)
            all_pair_stats.extend(pair_stats)
            if H_target_to_ref is None:
                homography_failed = True
        result['all_pair_stats'] = all_pair_stats
        if homography_failed:
            result.update(discard=True, discard_reason=fail_reason, status='DISCARD')
            return result

        H_contact = cumulative_H_list[0]
        result['H_contact_to_ref'] = H_contact
        mu_transformed = transform_points(contact_means.astype(np.float32), H_contact)
        if mu_transformed is None:
            result.update(discard=True, discard_reason='missing_transformed_contact_points', status='DISCARD')
            return result
        result['mu_transformed'] = mu_transformed
        contact_centroid = mu_transformed.mean(axis=0)
        result['contact_centroid'] = contact_centroid

        tau_transformed = []
        for offset, H in enumerate(cumulative_H_list):
            if H is not None:
                pt = transform_points(trajectory_pixels[offset].reshape(1, -1).astype(np.float32), H)
                tau_transformed.append(pt[0])
            else:
                tau_transformed.append(None)
        result['tau_transformed'] = tau_transformed

        frame_det_contact = self.detections[int(t_contact)]
        active_hand_bbox_norm = get_active_hand_bbox(frame_det_contact, active_hand, score_threshold=0.5)
        object_bboxes_norm = get_valid_object_bboxes(frame_det_contact, score_threshold=0.5)
        active_object_bbox_norm = select_active_object_bbox(active_hand_bbox_norm, object_bboxes_norm) if active_hand_bbox_norm is not None else None
        if active_object_bbox_norm is None:
            result.update(discard=True, discard_reason='no_active_object_bbox', status='DISCARD')
            return result
        active_object_bbox = np.array([
            active_object_bbox_norm[0] * w_img,
            active_object_bbox_norm[1] * h_img,
            active_object_bbox_norm[2] * w_img,
            active_object_bbox_norm[3] * h_img,
        ])

        object_polygon_ref = transform_bbox_to_polygon(active_object_bbox, H_contact)
        result['object_polygon_ref'] = object_polygon_ref
        original_area = (active_object_bbox[2] - active_object_bbox[0]) * (active_object_bbox[3] - active_object_bbox[1])
        projected_area = polygon_area(object_polygon_ref)
        area_ratio = projected_area / original_area if original_area > 0 else 0.0
        result['projected_area'] = projected_area
        result['area_ratio'] = area_ratio
        if projected_area <= 0 or area_ratio < 0.15:
            result.update(discard=True, discard_reason='projected_object_polygon_invalid', status='DISCARD')
            return result

        inside_count = count_points_near_polygon(mu_transformed, object_polygon_ref, tolerance_px=8.0)
        result['inside_count'] = inside_count
        if inside_count < 4:
            result.update(discard=True, discard_reason='transformed_contact_points_off_object', status='DISCARD')
            return result

        centroid_dist = cv2.pointPolygonTest(object_polygon_ref.astype(np.float32), tuple(contact_centroid), True)
        result['centroid_dist'] = centroid_dist
        if centroid_dist < -8.0:
            result.update(discard=True, discard_reason='transformed_contact_centroid_off_object', status='DISCARD')
            return result

        for i, pt in enumerate(tau_transformed):
            if pt is None:
                continue
            if pt[0] < 0 or pt[1] < 0 or pt[0] > w_img or pt[1] > h_img:
                result.update(discard=True, discard_reason=f'trajectory_out_of_bounds_t+{i}', status='DISCARD')
                return result
        for i, mu in enumerate(mu_transformed):
            if mu[0] < 0 or mu[1] < 0 or mu[0] > w_img or mu[1] > h_img:
                result.update(discard=True, discard_reason=f'contact_point_out_of_bounds_mu_{i+1}', status='DISCARD')
                return result

        hand_polygon_ref = None
        for hand in frame_det_contact.hands:
            if hand.score > 0.5 and hand.side.name.lower() == active_hand:
                hand_bbox = np.array([
                    hand.bbox.left * w_img,
                    hand.bbox.top * h_img,
                    hand.bbox.right * w_img,
                    hand.bbox.bottom * h_img,
                ])
                hand_polygon_ref = transform_bbox_to_polygon(hand_bbox, H_contact)
                break
        result['hand_polygon_ref'] = hand_polygon_ref

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        full_overlay = draw_problem3_full_overlay(
            ref_img,
            object_polygon_ref,
            hand_polygon_ref,
            mu_transformed,
            tau_transformed,
            int(t_contact),
            active_hand,
            ref_idx,
            result['discard'],
            result['discard_reason'],
            contact_centroid,
        )
        full_overlay_path = output_dir / 'problem3_ref_full_overlay.png'
        cv2.imwrite(str(full_overlay_path), full_overlay)
        result['full_overlay_path'] = str(full_overlay_path)

        crop_overlay, crop_bbox = draw_problem3_crop_overlay(
            ref_img,
            mu_transformed,
            tau_transformed,
            object_polygon_ref,
            crop_size=150,
        )
        crop_path = output_dir / 'problem3_ref_crop.png'
        cv2.imwrite(str(crop_path), cv2.cvtColor(crop_overlay, cv2.COLOR_RGB2BGR))
        result['crop_path'] = str(crop_path)
        result['crop_bbox'] = crop_bbox
        result['status'] = 'KEEP'
        return result


def smooth_contact_binary(values):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < 7:
        return values.astype(int)
    return (savgol_filter(values, window_length=7, polyorder=2) > 0.75).astype(int)


def build_contact_arrays(detections):
    left = np.zeros(len(detections), dtype=np.float32)
    right = np.zeros(len(detections), dtype=np.float32)
    for frame_idx, frame_det in enumerate(detections):
        if not hasattr(frame_det, 'hands'):
            continue
        for hand in frame_det.hands:
            if hand.score <= 0.5:
                continue
            is_contact = hand.state in [HandState.PORTABLE_OBJECT, HandState.STATIONARY_OBJECT]
            if not is_contact:
                continue
            side = hand.side.name.lower()
            if side == 'left':
                left[frame_idx] = 1.0
            elif side == 'right':
                right[frame_idx] = 1.0
    return smooth_contact_binary(left), smooth_contact_binary(right)


def get_subactions():
    df = pd.read_csv(ANNOTATION_CSV)
    subactions = df[df['video_id'] == SUBACTION_VIDEO_ID].copy()
    subactions = subactions.sort_values(['start_frame', 'stop_frame', 'narration_id']).head(NUM_SUBACTIONS)
    subactions = subactions.reset_index(drop=True)
    return subactions


def compute_subaction_frame_coverage(subactions):
    total_annotated_frames = int((subactions['stop_frame'] - subactions['start_frame'] + 1).sum())
    covered = set()
    for _, item in subactions.iterrows():
        start_0 = max(0, int(item['start_frame']) - 1)
        stop_0 = int(item['stop_frame']) - 1
        if start_0 <= stop_0:
            covered.update(range(start_0, stop_0 + 1))
    return {
        'total_annotated_frames': total_annotated_frames,
        'unique_covered_frames': int(len(covered)),
    }


def subaction_bounds(row, num_frames):
    start_0 = max(0, int(row['start_frame']) - 1)
    stop_0 = min(num_frames - 1, int(row['stop_frame']) - 1)
    return start_0, stop_0


def contact_candidates_for_subaction(row, left_binary, right_binary, strategy, num_frames, apply_cap=True):
    start_0, stop_0 = subaction_bounds(row, num_frames)
    if start_0 > stop_0:
        return []

    candidates = []
    for frame_idx in range(start_0, stop_0 + 1):
        if left_binary[frame_idx] == 1:
            candidates.append({'frame': int(frame_idx), 'hand': 'left'})
        if right_binary[frame_idx] == 1:
            candidates.append({'frame': int(frame_idx), 'hand': 'right'})

    candidates = sorted(candidates, key=lambda item: (item['frame'], item['hand']))
    if strategy == 'first_contact':
        return candidates[:1]
    if apply_cap and MAX_CANDIDATES_PER_SUBACTION is not None and len(candidates) > MAX_CANDIDATES_PER_SUBACTION:
        # Evenly sample across time so the diagnostic covers the whole subaction,
        # instead of spending all Cell3 time on near-duplicate adjacent frames.
        idxs = np.linspace(0, len(candidates) - 1, int(MAX_CANDIDATES_PER_SUBACTION))
        idxs = sorted(set(int(round(v)) for v in idxs))
        return [candidates[i] for i in idxs]
    return candidates


def min_ref_frame_for_strategy(row, reference_strategy):
    start_0 = max(0, int(row['start_frame']) - 1)
    if reference_strategy == 'inside_subaction':
        return start_0
    if reference_strategy == 'window_before_start_120':
        return max(0, start_0 - REFERENCE_WINDOW_BEFORE_START)
    if reference_strategy == 'full_previous':
        return 0
    raise ValueError(f'unknown reference_strategy: {reference_strategy}')


def compact_pair_stats(cell3_result):
    stats = cell3_result.get('all_pair_stats') or []
    ratios = []
    inliers = []
    good_matches = []
    fail_reasons = []
    for item in stats:
        if item.get('inlier_ratio') is not None:
            ratios.append(float(item.get('inlier_ratio', 0.0)))
        if item.get('inliers') is not None:
            inliers.append(int(item.get('inliers', 0)))
        if item.get('good_matches') is not None:
            good_matches.append(int(item.get('good_matches', 0)))
        if item.get('fail_reason'):
            fail_reasons.append(str(item.get('fail_reason')))
    return {
        'homography_pairs': int(len(stats)),
        'homography_min_inlier_ratio': min(ratios) if ratios else None,
        'homography_min_inliers': min(inliers) if inliers else None,
        'homography_min_good_matches': min(good_matches) if good_matches else None,
        'homography_fail_reasons': ';'.join(sorted(set(fail_reasons))) if fail_reasons else None,
    }


def score_success_candidate(record):
    ratio = record.get('homography_min_inlier_ratio')
    ratio = 0.0 if ratio is None or pd.isna(ratio) else float(ratio)
    contact_points = int(record.get('contact_points') or 0)
    traj_pts = int(record.get('trajectory_points') or 0)
    ref_gap = int(record.get('ref_gap') or 0)
    missing_traj = int(record.get('missing_trajectory_count') or 0)
    return 2.0 * ratio + 0.02 * contact_points + 0.5 * traj_pts - 0.01 * ref_gap - 0.5 * missing_traj


def write_cell4_pipeline_outputs(record, cell2_result, cell3_result, sample_dir):
    sample_dir = Path(sample_dir)
    sample_dir.mkdir(parents=True, exist_ok=True)

    ref_idx = int(cell3_result['ref_idx'])
    ref_path = IMAGE_DIR / f'frame_{ref_idx + 1:010d}.jpg'
    ref_img_bgr = cv2.imread(str(ref_path))
    if ref_img_bgr is None:
        raise FileNotFoundError(f'missing reference frame: {ref_path}')
    ref_img_rgb = cv2.cvtColor(ref_img_bgr, cv2.COLOR_BGR2RGB)
    h_img, w_img = ref_img_rgb.shape[:2]

    contact_means = cell2_result['contact_means']
    contact_weights = cell2_result['contact_weights']
    contact_covariances = cell2_result['contact_covariances']
    mu_transformed = np.asarray(cell3_result['mu_transformed'], dtype=np.float32)
    H_contact_to_ref = cell3_result['H_contact_to_ref']

    heatmap_mode = 'covariance'
    covariances_ref = transform_covariances_by_homography(
        contact_covariances,
        contact_means,
        H_contact_to_ref,
    )
    per_mode_heatmaps = build_label_heatmaps(
        image_shape=(h_img, w_img),
        centers_xy=mu_transformed,
        sigma_px=12.0,
        weights=contact_weights,
        normalize_each=True,
        covariances_xy=covariances_ref,
        covariance_scale=6.0,
        min_sigma_px=7.0,
        max_sigma_px=40.0,
    )
    merged_heatmap = merge_label_heatmaps(
        per_mode_heatmaps=per_mode_heatmaps,
        merge_method='sum',
        normalize_output=True,
    )

    heatmap_paths = silent_call(
        save_label_heatmap_outputs,
        merged_heatmap=merged_heatmap,
        ref_image=ref_img_rgb,
        output_dir=sample_dir,
        prefix='label_heatmap',
        sigma_px=12.0,
        merge_method='sum',
        use_weights=True,
        heatmap_mode=heatmap_mode,
        t_contact=record['frame_0_based'],
        active_hand=record['hand'],
        ref_idx=ref_idx,
    )

    reference_path = sample_dir / 'reference_frame.png'
    cv2.imwrite(str(reference_path), ref_img_bgr)

    contact_anchor = cell3_result.get('contact_centroid')
    if contact_anchor is None:
        contact_anchor = mu_transformed.mean(axis=0)
    vrb_style_img, arrow_info = draw_vrb_style_affordance_overlay(
        ref_img_rgb,
        merged_heatmap,
        cell3_result.get('tau_transformed', []),
        contact_anchor,
        heatmap_alpha=0.45,
        colormap=cv2.COLORMAP_JET,
        arrow_length_px=None,
        arrow_length_heatmap_ratio=1.2,
        arrow_width_px=None,
        source_step='last',
    )
    vrb_style_path = sample_dir / 'vrb_style_affordance.png'
    cv2.imwrite(str(vrb_style_path), cv2.cvtColor(vrb_style_img, cv2.COLOR_RGB2BGR))

    diagnostics = {
        'problem3_full_overlay': str(sample_dir / 'diagnostics' / 'problem3_ref_full_overlay.png'),
        'problem3_crop': str(sample_dir / 'diagnostics' / 'problem3_ref_crop.png'),
    }

    return {
        'sample_dir': str(sample_dir),
        'reference_frame': str(reference_path),
        'label_heatmap_npy': heatmap_paths['npy_path'],
        'label_heatmap_png': heatmap_paths['png_path'],
        'label_heatmap_overlay': heatmap_paths['overlay_path'],
        'vrb_style_affordance': str(vrb_style_path),
        'heatmap_mode': heatmap_mode,
        'heatmap_shape': tuple(int(v) for v in merged_heatmap.shape),
        'arrow_generated': arrow_info is not None,
        **diagnostics,
    }


def base_candidate_record(exp, row, candidate, start_0, stop_0, candidate_index):
    return {
        'experiment_id': exp['experiment_id'],
        'experiment_name': exp['name'],
        'contact_strategy': exp['contact_strategy'],
        'reference_strategy': exp['reference_strategy'],
        'subaction_index': int(row.name),
        'narration_id': row['narration_id'],
        'video_id': row['video_id'],
        'narration': row['narration'],
        'verb': row['verb'],
        'noun': row['noun'],
        'start_frame_1_based': int(row['start_frame']),
        'stop_frame_1_based': int(row['stop_frame']),
        'clipped_start_frame_0_based': int(start_0),
        'clipped_stop_frame_0_based': int(stop_0),
        'candidate_index': int(candidate_index),
        'frame_0_based': int(candidate['frame']),
        'frame_1_based': int(candidate['frame']) + 1,
        'hand': candidate['hand'],
        'status': 'pending',
        'passed_stage': 'candidate',
        'failed_stage': None,
        'fail_reason': None,
        'contact_points': 0,
        'ref_idx': None,
        'ref_frame_1_based': None,
        'ref_gap': None,
        'trajectory_points': 0,
        'missing_trajectory_count': None,
        'heatmap_mode': None,
        'sample_score': None,
        'sample_dir': None,
    }


def run_one_candidate(exp, row, candidate, candidate_index, detections, contact_config, problem3_runner):
    start_0, stop_0 = subaction_bounds(row, len(detections))
    record = base_candidate_record(exp, row, candidate, start_0, stop_0, candidate_index)
    min_ref_idx = min_ref_frame_for_strategy(row, exp['reference_strategy'])
    record['min_ref_frame_idx'] = int(min_ref_idx)

    sample = {
        'frame': int(candidate['frame']),
        'hand': candidate['hand'],
        'source': f"{exp['experiment_id']}_{exp['contact_strategy']}",
        'narration_id': row['narration_id'],
    }

    cell2 = fit_cell2_contact_gmm(
        detections=detections,
        sample=sample,
        image_dir=IMAGE_DIR,
        contact_extraction_config=contact_config,
    )
    record['contact_points'] = int(cell2.get('contact_points', 0))
    if not cell2['passed']:
        record.update(status='discard', failed_stage='Cell2_contact_gmm', fail_reason=cell2['reason'])
        return record

    record['passed_stage'] = 'Cell2_contact_gmm'

    sample_dir = (
        OUTPUT_ROOT / 'experiments' / exp['experiment_id'] / 'pipeline_outputs'
        / f"subaction_{int(row.name):02d}_{row['narration_id']}"
        / f"frame_{int(candidate['frame']):06d}_{candidate['hand']}"
    )
    diagnostics_dir = sample_dir / 'diagnostics'
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    cell3 = problem3_runner.run_cell3(
        t_contact=int(candidate['frame']),
        active_hand=candidate['hand'],
        contact_means=cell2['contact_means'],
        output_dir=diagnostics_dir,
        min_ref_frame_idx=min_ref_idx,
        discard=False,
        discard_reason=None,
    )

    record['ref_idx'] = cell3.get('ref_idx')
    if record['ref_idx'] is not None:
        record['ref_frame_1_based'] = int(record['ref_idx']) + 1
        record['ref_gap'] = int(candidate['frame']) - int(record['ref_idx'])
    record['trajectory_points'] = len(cell3.get('trajectory_pixels') or [])
    record['missing_trajectory_count'] = len(cell3.get('trajectory_missing_offsets') or [])
    record.update(compact_pair_stats(cell3))

    if cell3.get('H_contact_to_ref') is not None:
        record['passed_stage'] = 'Cell3_homography_available'
    if cell3.get('discard') or cell3.get('status') != 'KEEP':
        reason = cell3.get('discard_reason') or 'cell3_discard'
        record.update(status='discard', failed_stage='Cell3_homography_geometry', fail_reason=reason)
        return record

    record['passed_stage'] = 'Cell3_homography_geometry'

    cell4 = run_cell4_heatmap_gate(
        cell3_result=cell3,
        contact_means=cell2['contact_means'],
        contact_weights=cell2['contact_weights'],
        contact_covariances=cell2['contact_covariances'],
        image_dir=IMAGE_DIR,
    )
    if not cell4['passed']:
        record.update(status='discard', failed_stage='Cell4_label_heatmap', fail_reason=cell4['reason'])
        return record

    artifacts = write_cell4_pipeline_outputs(record, cell2, cell3, sample_dir)
    record.update(artifacts)
    record['status'] = 'keep'
    record['passed_stage'] = 'Cell4_label_heatmap'
    record['failed_stage'] = None
    record['fail_reason'] = None
    record['heatmap_mode'] = cell4.get('heatmap_mode')
    record['sample_score'] = score_success_candidate(record)

    candidate_json = sample_dir / 'candidate_result.json'
    candidate_json.write_text(json.dumps(_json_safe(record), ensure_ascii=False, indent=2), encoding='utf-8')
    return record


def summarize_subactions(candidate_df, subactions, exp):
    rows = []
    for idx, row in subactions.iterrows():
        sub_df = candidate_df[
            (candidate_df['experiment_id'] == exp['experiment_id'])
            & (candidate_df['subaction_index'] == idx)
        ]
        success_df = sub_df[sub_df['status'] == 'keep'].copy()
        best = None
        if not success_df.empty:
            success_df = success_df.sort_values('sample_score', ascending=False)
            best = success_df.iloc[0]
        fail_reason = None
        if not sub_df.empty:
            failed = sub_df[sub_df['status'] != 'keep']
            if not failed.empty:
                fail_reason = failed['fail_reason'].value_counts(dropna=True).index[0]
        else:
            fail_reason = 'no_contact_candidates'
        rows.append({
            'experiment_id': exp['experiment_id'],
            'experiment_name': exp['name'],
            'subaction_index': int(idx),
            'narration_id': row['narration_id'],
            'narration': row['narration'],
            'frames_1_based': f"{int(row['start_frame'])}-{int(row['stop_frame'])}",
            'candidates': int(len(sub_df)),
            'cell2_pass': int((sub_df['passed_stage'].isin(['Cell2_contact_gmm', 'Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'ref_found': int(sub_df['ref_idx'].notna().sum()) if not sub_df.empty else 0,
            'homography_available': int((sub_df['passed_stage'].isin(['Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'geometry_pass': int((sub_df['passed_stage'].isin(['Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'heatmap_pass': int((sub_df['status'] == 'keep').sum()) if not sub_df.empty else 0,
            'final_samples': int((sub_df['status'] == 'keep').sum()) if not sub_df.empty else 0,
            'best_frame_0_based': int(best['frame_0_based']) if best is not None else None,
            'best_ref_idx': int(best['ref_idx']) if best is not None and pd.notna(best['ref_idx']) else None,
            'best_score': float(best['sample_score']) if best is not None else None,
            'best_sample_dir': best['sample_dir'] if best is not None else None,
            'main_failure': fail_reason,
        })
    return rows


def build_overview(candidate_df, subaction_summary_df):
    overview = []
    for exp in EXPERIMENTS:
        exp_candidates = candidate_df[candidate_df['experiment_id'] == exp['experiment_id']]
        exp_sub = subaction_summary_df[subaction_summary_df['experiment_id'] == exp['experiment_id']]
        failure_counts = exp_candidates[exp_candidates['status'] != 'keep']['fail_reason'].value_counts(dropna=True)
        main_failure = failure_counts.index[0] if len(failure_counts) else None
        overview.append({
            'experiment_id': exp['experiment_id'],
            'experiment': exp['name'],
            'contact_strategy': exp['contact_strategy'],
            'reference_strategy': exp['reference_strategy'],
            'subactions_total': int(NUM_SUBACTIONS),
            'subactions_success': int((exp_sub['final_samples'] > 0).sum()),
            'candidates_total': int(len(exp_candidates)),
            'cell2_pass': int((exp_candidates['passed_stage'].isin(['Cell2_contact_gmm', 'Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'ref_found': int(exp_candidates['ref_idx'].notna().sum()),
            'homography_available': int((exp_candidates['passed_stage'].isin(['Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'geometry_pass': int((exp_candidates['passed_stage'].isin(['Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'heatmap_pass': int((exp_candidates['status'] == 'keep').sum()),
            'main_failure': main_failure,
            'description': exp['description'],
        })
    return pd.DataFrame(overview)


def save_bar_chart_failure_reasons(candidate_df, path):
    rows = []
    for exp in EXPERIMENTS:
        exp_failed = candidate_df[(candidate_df['experiment_id'] == exp['experiment_id']) & (candidate_df['status'] != 'keep')]
        for reason, count in exp_failed['fail_reason'].value_counts(dropna=True).items():
            rows.append({'experiment_id': exp['experiment_id'], 'reason': reason, 'count': int(count)})
    if not rows:
        return
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index='reason', columns='experiment_id', values='count', fill_value=0, aggfunc='sum')
    ax = pivot.plot(kind='barh', figsize=(12, max(4, 0.35 * len(pivot))))
    ax.set_title('Failure reason counts by experiment')
    ax.set_xlabel('candidate count')
    ax.set_ylabel('failure reason')
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def save_funnel_chart(overview_df, path):
    stages = ['candidates_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass']
    labels = ['candidates', 'Cell2', 'reference', 'homography', 'geometry', 'heatmap']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(labels))
    for _, row in overview_df.iterrows():
        y = [int(row[s]) for s in stages]
        ax.plot(x, y, marker='o', label=row['experiment_id'])
        for xi, yi in zip(x, y):
            ax.text(xi, yi, str(yi), ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('candidate count')
    ax.set_title('Pipeline funnel by experiment')
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def save_timeline_chart(candidate_df, subactions, path, experiment_id='E2'):
    exp_df = candidate_df[candidate_df['experiment_id'] == experiment_id]
    fig, ax = plt.subplots(figsize=(14, max(5, 0.5 * len(subactions))))
    y_ticks = []
    y_labels = []
    for idx, row in subactions.iterrows():
        y = len(subactions) - idx
        y_ticks.append(y)
        y_labels.append(f"{idx}: {row['narration_id']} {row['narration']}")
        start = int(row['start_frame']) - 1
        stop = int(row['stop_frame']) - 1
        ax.hlines(y, start, stop, color='#9aa0a6', linewidth=7, alpha=0.45)
        sub_df = exp_df[exp_df['subaction_index'] == idx]
        failed = sub_df[sub_df['status'] != 'keep']
        kept = sub_df[sub_df['status'] == 'keep']
        ax.scatter(failed['frame_0_based'], [y] * len(failed), s=14, color='#d93025', alpha=0.55, label='failed candidates' if idx == 0 else None)
        ax.scatter(kept['frame_0_based'], [y] * len(kept), s=42, color='#188038', marker='o', label='success samples' if idx == 0 else None)
        for _, item in kept.iterrows():
            if pd.notna(item.get('ref_idx')):
                ax.annotate('', xy=(item['frame_0_based'], y + 0.08), xytext=(item['ref_idx'], y + 0.08), arrowprops=dict(arrowstyle='->', color='#1a73e8', lw=1.0, alpha=0.75))
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel('0-based frame index')
    ax.set_title(f'Timeline: {experiment_id} candidates, successes, and reference arrows')
    ax.grid(True, axis='x', alpha=0.25)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def markdown_table(df, columns):
    if df.empty:
        return 'None\n'
    view = df[columns].copy()
    view = view.where(pd.notna(view), '')
    headers = [str(col) for col in columns]
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for _, row in view.iterrows():
        values = [str(row[col]).replace('\n', ' ') for col in columns]
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)


def write_markdown_report(overview_df, subaction_summary_df, candidate_df, output_root, frame_coverage):
    e2_sub = subaction_summary_df[subaction_summary_df['experiment_id'] == 'E2'].copy()
    failure_rows = []
    for exp in EXPERIMENTS:
        failed = candidate_df[(candidate_df['experiment_id'] == exp['experiment_id']) & (candidate_df['status'] != 'keep')]
        counts = failed['fail_reason'].value_counts(dropna=True)
        top = ', '.join([f'{reason}: {count}' for reason, count in counts.head(5).items()]) if len(counts) else 'None'
        failure_rows.append({'experiment_id': exp['experiment_id'], 'top_failures': top})
    failure_df = pd.DataFrame(failure_rows)

    e0 = overview_df[overview_df['experiment_id'] == 'E0'].iloc[0]
    e1 = overview_df[overview_df['experiment_id'] == 'E1'].iloc[0]
    e2 = overview_df[overview_df['experiment_id'] == 'E2'].iloc[0]
    e3 = overview_df[overview_df['experiment_id'] == 'E3'].iloc[0]
    conclusions = []
    if e1['subactions_success'] > e0['subactions_success'] or e1['heatmap_pass'] > e0['heatmap_pass']:
        conclusions.append('E1 比 E0 有提升：reference 被 subaction 起点限制是主要问题之一。')
    else:
        conclusions.append('E1 相比 E0 没有明显提升：仅放宽 reference 还不够，需继续看 candidate frame 或 homography/geometry。')
    if e2['subactions_success'] > e1['subactions_success'] or e2['heatmap_pass'] > e1['heatmap_pass']:
        conclusions.append('E2 比 E1 有提升：只取 first contact 会漏掉 subaction 内更可用的帧。')
    else:
        conclusions.append('E2 相比 E1 没有明显提升：当前 subaction 内其他 contact frames 也没有解决主要瓶颈。')
    if e3['heatmap_pass'] > e2['heatmap_pass']:
        conclusions.append('E3 比 E2 还有提升：120 帧 reference 窗口可能偏窄。')
    else:
        conclusions.append('E3 相比 E2 提升不明显：120 帧 reference 窗口基本够用，剩余问题更可能在 homography/geometry/检测质量。')

    lines = [
        '# 数据处理小批量测试前100个subaction - 诊断实验报告',
        '',
        f'- video_id: `{SUBACTION_VIDEO_ID}`',
        f'- subactions: first `{NUM_SUBACTIONS}` annotations of this video',
        f'- output root: `{output_root}`',
        f'- reference window before subaction start: `{REFERENCE_WINDOW_BEFORE_START}` frames',
        f'- all-contact deep-run cap per subaction: `{MAX_CANDIDATES_PER_SUBACTION}` time-uniform candidates',
        f'- annotated frame intervals total: `{frame_coverage["total_annotated_frames"]}` frames',
        f'- unique covered frames after overlap removal: `{frame_coverage["unique_covered_frames"]}` frames',
        '',
        '## 怎么读这个报告',
        '',
        f'- `subactions_success` 表示 {NUM_SUBACTIONS} 个 subaction 里有多少个至少生成了 1 个完整 heatmap/trajectory 样本。',
        '- `heatmap_pass` 表示候选帧级别最终成功样本数。',
        '- E2/E3 的 all-contact 候选采用时间均匀抽样深跑，避免大量相邻帧重复做昂贵的 homography。',
        '- `ref_found` 低，说明主要卡在 human-less reference frame。',
        '- `homography_available / geometry_pass` 低，说明找到 reference 后投影或几何一致性不过。',
        '',
        '## 结论速览',
        '',
    ]
    lines += [f'- {item}' for item in conclusions]
    lines += [
        '',
        '## 实验总览',
        '',
        markdown_table(overview_df, ['experiment_id', 'subactions_success', 'candidates_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']),
        '',
        '## 主方案 E2 的 subaction 级结果',
        '',
        markdown_table(e2_sub, ['subaction_index', 'narration_id', 'narration', 'frames_1_based', 'candidates', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'best_frame_0_based', 'best_ref_idx', 'main_failure']),
        '',
        '## 主要失败原因',
        '',
        markdown_table(failure_df, ['experiment_id', 'top_failures']),
        '',
        '## 图表',
        '',
        '- pipeline funnel: `charts/pipeline_funnel.png`',
        '- failure reasons: `charts/failure_reasons.png`',
        '- E2 timeline: `charts/timeline_E2.png`',
        '',
        '## 明细文件',
        '',
        '- all results JSON: `experiment_results.json`',
        '- overview CSV: `experiment_overview.csv`',
        '- subaction summary CSV: `subaction_summary.csv`',
        '- candidate diagnostics CSV: `candidate_diagnostics.csv`',
        '- successful pipeline outputs: `experiments/<E*>/pipeline_outputs/`',
    ]
    report_path = output_root / 'summary.md'
    report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return report_path


def run_subaction_diagnostic_experiments():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    charts_dir = OUTPUT_ROOT / 'charts'
    charts_dir.mkdir(parents=True, exist_ok=True)

    print(f'Loading HOA detections: {HOA_PKL}')
    detections = load_detections(str(HOA_PKL))
    print(f'Total frames: {len(detections)}')

    print(f'Loading subactions: {ANNOTATION_CSV}')
    subactions = get_subactions()
    print(f'Selected subactions: {len(subactions)}')
    frame_coverage = compute_subaction_frame_coverage(subactions)
    print(f"Annotated frame intervals total: {frame_coverage['total_annotated_frames']}")
    print(f"Unique covered frames: {frame_coverage['unique_covered_frames']}")
    display(subactions[['narration_id', 'narration', 'start_frame', 'stop_frame', 'verb', 'noun']])

    print('Computing smoothed contact arrays...')
    left_binary, right_binary = build_contact_arrays(detections)

    contact_config = ContactExtractionConfig(
        use_object_mask=True,
        use_object_boundary=True,
        boundary_distance_px=6.0,
        project_points_to_object_boundary=True,
    )
    problem3_runner = Problem3CachedRunner(detections, IMAGE_DIR, OUTPUT_ROOT)

    all_records = []
    candidate_plan_rows = []

    for exp in EXPERIMENTS:
        print('\n' + '=' * 88)
        print(f"Running {exp['experiment_id']}: {exp['description']}")
        print('=' * 88)
        for idx, row in subactions.iterrows():
            start_0, stop_0 = subaction_bounds(row, len(detections))
            raw_candidates = contact_candidates_for_subaction(
                row,
                left_binary=left_binary,
                right_binary=right_binary,
                strategy=exp['contact_strategy'],
                num_frames=len(detections),
                apply_cap=False,
            )
            candidates = contact_candidates_for_subaction(
                row,
                left_binary=left_binary,
                right_binary=right_binary,
                strategy=exp['contact_strategy'],
                num_frames=len(detections),
                apply_cap=True,
            )
            candidate_plan_rows.append({
                'experiment_id': exp['experiment_id'],
                'subaction_index': int(idx),
                'narration_id': row['narration_id'],
                'narration': row['narration'],
                'start_frame_0_based': int(start_0),
                'stop_frame_0_based': int(stop_0),
                'raw_candidate_count': int(len(raw_candidates)),
                'deep_run_candidate_count': int(len(candidates)),
            })
            print(f"{exp['experiment_id']} subaction {idx:02d} {row['narration_id']} | {row['narration']} | raw_candidates={len(raw_candidates)} | deep_run={len(candidates)}")
            if not candidates:
                empty_record = {
                    'experiment_id': exp['experiment_id'],
                    'experiment_name': exp['name'],
                    'contact_strategy': exp['contact_strategy'],
                    'reference_strategy': exp['reference_strategy'],
                    'subaction_index': int(idx),
                    'narration_id': row['narration_id'],
                    'video_id': row['video_id'],
                    'narration': row['narration'],
                    'verb': row['verb'],
                    'noun': row['noun'],
                    'start_frame_1_based': int(row['start_frame']),
                    'stop_frame_1_based': int(row['stop_frame']),
                    'clipped_start_frame_0_based': int(start_0),
                    'clipped_stop_frame_0_based': int(stop_0),
                    'candidate_index': None,
                    'frame_0_based': None,
                    'frame_1_based': None,
                    'hand': None,
                    'status': 'discard',
                    'passed_stage': 'no_candidate',
                    'failed_stage': 'subaction_contact',
                    'fail_reason': 'no_smoothed_contact_in_subaction',
                    'contact_points': 0,
                    'ref_idx': None,
                    'ref_frame_1_based': None,
                    'ref_gap': None,
                    'trajectory_points': 0,
                    'missing_trajectory_count': None,
                    'heatmap_mode': None,
                    'sample_score': None,
                    'sample_dir': None,
                    'min_ref_frame_idx': min_ref_frame_for_strategy(row, exp['reference_strategy']),
                }
                all_records.append(empty_record)
                continue
            for candidate_index, candidate in enumerate(candidates):
                record = run_one_candidate(exp, row, candidate, candidate_index, detections, contact_config, problem3_runner)
                all_records.append(record)

    candidate_df = pd.DataFrame(all_records)
    subaction_rows = []
    for exp in EXPERIMENTS:
        subaction_rows.extend(summarize_subactions(candidate_df, subactions, exp))
    subaction_summary_df = pd.DataFrame(subaction_rows)
    overview_df = build_overview(candidate_df, subaction_summary_df)
    candidate_plan_df = pd.DataFrame(candidate_plan_rows)

    json_path = OUTPUT_ROOT / 'experiment_results.json'
    overview_path = OUTPUT_ROOT / 'experiment_overview.csv'
    subaction_path = OUTPUT_ROOT / 'subaction_summary.csv'
    candidate_path = OUTPUT_ROOT / 'candidate_diagnostics.csv'
    candidate_plan_path = OUTPUT_ROOT / 'candidate_plan.csv'

    json_payload = {
        'video_id': SUBACTION_VIDEO_ID,
        'num_subactions': NUM_SUBACTIONS,
        'reference_window_before_start': REFERENCE_WINDOW_BEFORE_START,
        'frame_coverage': frame_coverage,
        'experiments': EXPERIMENTS,
        'overview': overview_df.to_dict(orient='records'),
        'subaction_summary': subaction_summary_df.to_dict(orient='records'),
        'candidate_diagnostics': candidate_df.to_dict(orient='records'),
        'candidate_plan': candidate_plan_df.to_dict(orient='records'),
        'problem3_cache_stats': dict(problem3_runner.cache_stats),
    }
    json_path.write_text(json.dumps(_json_safe(json_payload), ensure_ascii=False, indent=2), encoding='utf-8')
    overview_df.to_csv(overview_path, index=False)
    subaction_summary_df.to_csv(subaction_path, index=False)
    candidate_df.to_csv(candidate_path, index=False)
    candidate_plan_df.to_csv(candidate_plan_path, index=False)

    save_funnel_chart(overview_df, charts_dir / 'pipeline_funnel.png')
    save_bar_chart_failure_reasons(candidate_df, charts_dir / 'failure_reasons.png')
    save_timeline_chart(candidate_df, subactions, charts_dir / 'timeline_E2.png', experiment_id='E2')
    report_path = write_markdown_report(overview_df, subaction_summary_df, candidate_df, OUTPUT_ROOT, frame_coverage)

    print('\nExperiment overview')
    display(overview_df[['experiment_id', 'subactions_success', 'candidates_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']])
    print(f'JSON: {json_path}')
    print(f'Overview CSV: {overview_path}')
    print(f'Subaction summary CSV: {subaction_path}')
    print(f'Candidate diagnostics CSV: {candidate_path}')
    print(f'Summary: {report_path}')
    print(f'Pipeline outputs: {OUTPUT_ROOT / "experiments"}')
    print(f'Problem3 cache stats: {dict(problem3_runner.cache_stats)}')

    return {
        'overview': overview_df,
        'subaction_summary': subaction_summary_df,
        'candidate_diagnostics': candidate_df,
        'candidate_plan': candidate_plan_df,
        'frame_coverage': frame_coverage,
        'output_root': OUTPUT_ROOT,
        'summary_path': report_path,
        'problem3_cache_stats': dict(problem3_runner.cache_stats),
    }


subaction_diagnostic_result = run_subaction_diagnostic_experiments()


In [ ]:
# Cell 6: 查看诊断实验结果和主方案 E2 的成功样本
# ============================================================
# 这个 Cell 不重新跑 pipeline，只读取 Cell 5 写出的结果，方便快速查看。
# ============================================================

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

OUTPUT_ROOT = Path('outputs') / '数据处理小批量测试前100个subaction'
summary_path = OUTPUT_ROOT / 'summary.md'
overview_path = OUTPUT_ROOT / 'experiment_overview.csv'
subaction_path = OUTPUT_ROOT / 'subaction_summary.csv'
candidate_path = OUTPUT_ROOT / 'candidate_diagnostics.csv'

if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))
else:
    print('summary.md not found. Run Cell 5 first.')

if overview_path.exists():
    overview_df = pd.read_csv(overview_path)
    print('Experiment overview')
    display(overview_df[['experiment_id', 'subactions_success', 'candidates_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']])

if subaction_path.exists():
    subaction_df = pd.read_csv(subaction_path)
    e2_subactions = subaction_df[subaction_df['experiment_id'] == 'E2']
    print('E2 subaction summary')
    display(e2_subactions[['subaction_index', 'narration_id', 'narration', 'frames_1_based', 'candidates', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'best_frame_0_based', 'best_ref_idx', 'main_failure']])

if candidate_path.exists():
    candidate_df = pd.read_csv(candidate_path)
    e2_success = candidate_df[(candidate_df['experiment_id'] == 'E2') & (candidate_df['status'] == 'keep')].copy()
    if not e2_success.empty:
        e2_success = e2_success.sort_values(['subaction_index', 'sample_score'], ascending=[True, False])
        print('E2 successful pipeline outputs')
        display(e2_success[['subaction_index', 'narration_id', 'frame_0_based', 'hand', 'ref_idx', 'contact_points', 'sample_score', 'sample_dir', 'label_heatmap_overlay', 'vrb_style_affordance']])
    else:
        print('E2 has no successful samples.')
